## Setup

In [47]:
from pathlib import Path
import pandas as pd
import numpy as np

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "README.md").exists() and (REPO_ROOT.parent / "README.md").exists():
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "outputs"
REPORTS_DIR = REPO_ROOT / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)


## Load Data

Read each provided table. Keep the original row meaning in mind before joining tables.

In [48]:
movies = pd.read_csv(DATA_DIR / 'movies_metadata.csv')
ratings = pd.read_csv(DATA_DIR / 'ratings.csv', parse_dates=['rating_datetime'])
links = pd.read_csv(DATA_DIR / 'links.csv')
credits = pd.read_csv(DATA_DIR / 'credits_curated.csv')
keywords = pd.read_csv(DATA_DIR / 'keywords_curated.csv')

tables = {
    'movies': movies,
    'ratings': ratings,
    'links': links,
    'credits': credits,
    'keywords': keywords,
}

## Basic Data Profile

Use this section for your first pass through row counts, columns, data types, missingness, duplicates, and suspicious values.

In [49]:
for name, df in tables.items():
    print(f'\n{name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
    display(df.head(3))


movies: 9,082 rows x 21 columns


,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year
0,1,862,tt0114709,Toy Story,Toy Story,en,1995-10-30,81.0,30000000,373554033.0,21.946943,7.7,5415.0,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Animation|Comedy|Family,Animation,1995
1,2,8844,tt0113497,Jumanji,Jumanji,en,1995-12-15,104.0,65000000,262797249.0,17.015539,6.9,2413.0,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...","[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Adventure|Fantasy|Family,Adventure,1995
2,3,15602,tt0113228,Grumpier Old Men,Grumpier Old Men,en,1995-12-22,101.0,0,0.0,11.712900,6.5,92.0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...","[{'name': 'Warner Bros.', 'id': 6194}, {'name'...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Romance|Comedy,Romance,1995



ratings: 99,810 rows x 5 columns


,userId,movieId,rating,timestamp,rating_datetime
0,1,31,2.5,1260759144,2009-12-14 02:52:24
1,1,1029,3.0,1260759179,2009-12-14 02:52:59
2,1,1061,3.0,1260759182,2009-12-14 02:53:02



links: 9,125 rows x 3 columns


,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0



credits: 9,082 rows x 3 columns


,tmdbId,director,top_cast
0,862,John Lasseter,Tom Hanks|Tim Allen|Don Rickles
1,8844,Joe Johnston,Robin Williams|Jonathan Hyde|Kirsten Dunst
2,15602,Howard Deutch,Walter Matthau|Jack Lemmon|Ann-Margret



keywords: 9,082 rows x 2 columns


,tmdbId,keywords_list
0,862,jealousy|toy|boy|friendship|friends|rivalry|bo...
1,8844,board game|disappearance|based on children's b...
2,15602,fishing|best friend|duringcreditsstinger|old men


In [50]:
profile_rows = []
for name, df in tables.items():
    profile_rows.append({
        'table': name,
        'rows': len(df),
        'columns': df.shape[1],
        'duplicate_rows': int(df.duplicated().sum()),
        'missing_cells': int(df.isna().sum().sum()),
    })
pd.DataFrame(profile_rows)

,table,rows,columns,duplicate_rows,missing_cells
0,movies,9082,21,0,72
1,ratings,99810,5,0,0
2,links,9125,3,0,13
3,credits,9082,3,0,114
4,keywords,9082,2,0,761


### Key and Granularity Audit

Use the expected key for each table before planning joins. `ratings` is the exception: its row meaning is a user-movie rating event, so check the combined `userId`-`movieId` key rather than expecting `movieId` alone to be unique. `links` is a crosswalk; because `movies` already contains both IDs, use it for validation unless your analysis needs one of its fields.


In [51]:
key_audit = pd.DataFrame([
    {"table": "movies", "expected_key": "movieId", "duplicate_keys": movies["movieId"].duplicated().sum(), "missing_key_values": movies["movieId"].isna().sum()},
    {"table": "movies", "expected_key": "tmdbId", "duplicate_keys": movies["tmdbId"].duplicated().sum(), "missing_key_values": movies["tmdbId"].isna().sum()},
    {"table": "ratings", "expected_key": "userId + movieId", "duplicate_keys": ratings[["userId", "movieId"]].duplicated().sum(), "missing_key_values": ratings[["userId", "movieId"]].isna().sum().sum()},
    {"table": "links", "expected_key": "movieId", "duplicate_keys": links["movieId"].duplicated().sum(), "missing_key_values": links["movieId"].isna().sum()},
    {"table": "credits", "expected_key": "tmdbId", "duplicate_keys": credits["tmdbId"].duplicated().sum(), "missing_key_values": credits["tmdbId"].isna().sum()},
    {"table": "keywords", "expected_key": "tmdbId", "duplicate_keys": keywords["tmdbId"].duplicated().sum(), "missing_key_values": keywords["tmdbId"].isna().sum()},
])

key_audit


,table,expected_key,duplicate_keys,missing_key_values
0,movies,movieId,0,0
1,movies,tmdbId,0,0
2,ratings,userId + movieId,0,0
3,links,movieId,0,0
4,credits,tmdbId,0,0
5,keywords,tmdbId,0,0


In [52]:
print(f"Total movies: {len(movies):,}")


Total movies: 9,082


### Do production budget and release timing (month or season) have a measurable effect on box office revenue and return on investment (ROI)?

In [53]:
columns = ['budget', 'revenue', 'release_date', 'movieId', 'title']

print(movies[columns].isna().sum())


budget          0
revenue         0
release_date    0
movieId         0
title           0
dtype: int64


In [54]:
# Make a copy of movies DataFrame
movies_clean = movies.copy()

# Clean the movies DataFrame by removing missing values from movieId, release_date, and title
movies_clean = movies_clean.dropna(subset=["movieId", "release_date", "title"])

# Convert release_date to a valid datetime format
movies_clean["release_date"] = pd.to_datetime(movies_clean["release_date"], errors="coerce")

# Remove movies with 0 budgets or revenues 
movies_clean = movies_clean[(movies_clean["budget"] != 0) & (movies_clean["revenue"] != 0)]

# Remove movies with a budget lesser than 1000
movies_clean = movies_clean[movies_clean["budget"] >= 1000]

# Remove movies with a revenue lesser than 10000
movies_clean = movies_clean[movies_clean["revenue"] >= 10000]

# Display the cleaned movies DataFrame
movies_clean.head()



,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year
0,1,862,tt0114709,Toy Story,Toy Story,en,1995-10-30,81.0,30000000,373554033.0,21.946943,7.7,5415.0,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Animation|Comedy|Family,Animation,1995
1,2,8844,tt0113497,Jumanji,Jumanji,en,1995-12-15,104.0,65000000,262797249.0,17.015539,6.9,2413.0,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...","[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Adventure|Fantasy|Family,Adventure,1995
3,4,31357,tt0114885,Waiting to Exhale,Waiting to Exhale,en,1995-12-22,127.0,16000000,81452156.0,3.859495,6.1,34.0,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",[{'name': 'Twentieth Century Fox Film Corporat...,"[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Comedy|Drama|Romance,Comedy,1995
5,6,949,tt0113277,Heat,Heat,en,1995-12-15,170.0,60000000,187436818.0,17.924927,7.7,1886.0,"[{'id': 28, 'name': 'Action'}, {'id': 80, 'nam...","[{'name': 'Regency Enterprises', 'id': 508}, {...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Action|Crime|Drama|Thriller,Action,1995
8,9,9091,tt0114576,Sudden Death,Sudden Death,en,1995-12-22,106.0,35000000,64350171.0,5.231580,5.5,174.0,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...","[{'name': 'Universal Pictures', 'id': 33}, {'n...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Action|Adventure|Thriller,Action,1995


In [55]:
# Create a table that shows movieId, title, release_date, runtime, budget, revenue, profit, ROI [(revenue-budget)/budget], profitable (yes or no), add a column for seasons, and yes or no column for summer or winter

movies_abt= (
    movies_clean
    .groupby("movieId")
    .agg(
        title=("title", "first"),
        release_date=("release_date", "first"),
        runtime=("runtime", "first"),
        budget=("budget", "first"),
        revenue=("revenue", "first"),
    )
    .reset_index()
    .assign(
        profit=lambda x: x["revenue"] - x["budget"],
        ROI=lambda x: x["profit"] / x["budget"],
        profitable=lambda x: x["ROI"].gt(0).map({True: "yes", False: "no"}),
        seasons=lambda x: x["release_date"].dt.month.map(
            {1: "Winter", 2: "Winter", 3: "Spring", 4: "Spring", 5: "Spring", 6: "Summer",
             7: "Summer", 8: "Summer", 9: "Fall", 10: "Fall", 11: "Fall", 12: "Winter"}
        ),
        summer_or_winter=lambda x: x["seasons"].isin(["Summer", "Winter"]).map(
            {True: "yes", False: "no"}
        ),
    )
)

# Display the resulting table
movies_abt.head()


,movieId,title,release_date,runtime,budget,revenue,profit,ROI,profitable,seasons,summer_or_winter
0,1,Toy Story,1995-10-30,81.0,30000000,373554033.0,343554033.0,11.451801,yes,Fall,no
1,2,Jumanji,1995-12-15,104.0,65000000,262797249.0,197797249.0,3.043035,yes,Winter,yes
2,4,Waiting to Exhale,1995-12-22,127.0,16000000,81452156.0,65452156.0,4.090760,yes,Winter,yes
3,6,Heat,1995-12-15,170.0,60000000,187436818.0,127436818.0,2.123947,yes,Winter,yes
4,9,Sudden Death,1995-12-22,106.0,35000000,64350171.0,29350171.0,0.838576,yes,Winter,yes


In [56]:
print("Movies",movies.shape)
print("Movies_abt", movies_abt.shape)

Movies (9082, 21)
Movies_abt (3819, 11)


In [59]:
# Create a data dictionary for movies_abt 
data_dict = {
    "Column": [
        "movieID",
        "title",
        "release_date",
        "runtime",
        "budget",
        "revenue",
        "profit",
        "ROI",
        "profitable",
        "seasons",
        "summer_or_winter"
    ],
    "Description": [
        "Unique identifier for each movie",
        "Title of the movie",
        "Original release date of the movie",
        "Movie runtime in minutes",
        "Production budget in dollars",
        "Total revenue generated in dollars",
        "Revenue minus production budget",
        "Return on investment calculated as profit divided by budget",
        "Indicates whether the movie generated a profit",
        "Season in which the movie was released",
        "Indicates whether the movie was released in summer or winter"
    ]
}
data_dictionary = pd.DataFrame(data_dict)

data_dictionary["Data Type"] = (
    data_dictionary["Column"]
    .map(movies_abt.dtypes.astype(str))
)


display(data_dictionary)

,Column,Description,Data Type
0,movieID,Unique identifier for each movie,NaN
1,title,Title of the movie,object
2,release_date,Original release date of the movie,datetime64[ns]
3,runtime,Movie runtime in minutes,float64
4,budget,Production budget in dollars,int64
5,revenue,Total revenue generated in dollars,float64
6,profit,Revenue minus production budget,float64
7,ROI,Return on investment calculated as profit divi...,float64
8,profitable,Indicates whether the movie generated a profit,object
9,seasons,Season in which the movie was released,object


In [60]:
# Table that shows the means, standard deviation, minimum, maximum, median, and skew of budget, revenue, profit, ROI, runtime named 'Key Numeric Variables'
movies_abt[["budget", "revenue", "profit", "ROI", "runtime"]].agg(
    ["mean", "std", "min", "max", "median", "skew"]
).rename_axis("Key Numeric Variables")

,budget,revenue,profit,ROI,runtime
Key Numeric Variables,,,,,
mean,3.695950e+07,1.127282e+08,7.576872e+07,9.588242,110.277298
std,4.306080e+07,1.819208e+08,1.540850e+08,220.429856,20.553252
min,6.000000e+03,1.028900e+04,-1.657101e+08,-0.999486,62.000000
max,3.800000e+08,2.787965e+09,2.550965e+09,12889.386667,248.000000
median,2.200000e+07,4.717504e+07,2.237795e+07,1.409742,106.000000
skew,2.238330e+00,4.102622e+00,4.547702e+00,54.017450,1.381560


In [61]:
# Table that explains the release season count, share, median budget, median revenue, median ROI, % profitiable named 'Budget and Return by Release Season'
movies_abt.groupby("seasons").agg(
    count=("movieId", "count"),
    share=("movieId", lambda x: x.count() / len(movies_abt) * 100),
    median_budget=("budget", "median"),
    median_revenue=("revenue", "median"),
    median_ROI=("ROI", "median"),
    profitable_pct=("profitable", lambda x: (x == "yes").mean() * 100),
).reset_index().rename(columns={"seasons": "season"}).set_index("season").rename_axis("Budget and Return by Release Season")

,count,share,median_budget,median_revenue,median_ROI,profitable_pct
Budget and Return by Release Season,,,,,,
Fall,1056,27.651218,20000000.0,36561377.5,1.086447,71.022727
Spring,854,22.361875,23500000.0,48734523.0,1.384459,78.220141
Summer,1024,26.813302,25000000.0,61474123.5,1.664058,81.250000
Winter,885,23.173606,23000000.0,49469904.0,1.507167,77.740113


In [62]:
# Column that shows the budget quartiles
movies_abt["budget_quartile"] = pd.qcut(
    movies_abt["budget"],
    q=4,
    labels=["Q1 (lowest)", "Q2", "Q3", "Q4 (highest)"],
    duplicates="drop",
)

# Column that shows the revenue quartile
movies_abt["revenue_quartile"] = pd.qcut(
    movies_abt["revenue"],
    q=4,
    labels=["Q1 (lowest)", "Q2", "Q3", "Q4 (highest)"],
    duplicates="drop",
)

# Column that shows the ROI quartile
movies_abt["ROI_quartile"] = pd.qcut(
    movies_abt["ROI"],
    q=4,
    labels=["Q1 (lowest)", "Q2", "Q3", "Q4 (highest)"],
    duplicates="drop",
)

# Table that shows the distribution of movies in each quartile
quartile_table = movies_abt.groupby(
    "budget_quartile",
    observed=True
).agg(
    min_budget=("budget", "min"),
    max_budget=("budget", "max"),
    Films=("budget", "count"),
    median_revenue=("revenue", "median"),
    median_ROI=("ROI", "median")
).reset_index()

# Rename the quartile column to Factor
quartile_table = quartile_table.rename(
    columns={"budget_quartile": "Factor"}
)

display(quartile_table)

,Factor,min_budget,max_budget,Films,median_revenue,median_ROI
0,Q1 (lowest),6000,8000000,980,10368693.5,3.000000
1,Q2,8045760,22000000,933,32482682.0,1.278315
2,Q3,22500000,50000000,1020,65048657.5,0.991988
3,Q4 (highest),50100000,380000000,886,200930070.5,1.292822
